# TopConNet v3 — Kaggle GNSS LOS/NLOS Classification

**Dataset:** Kaggle GNSS Classification (Hong Kong, 74k samples, ray-tracing labels)  
**Architecture:** Transformer Encoder on per-SV time-series  
**Improvements over v2:**
- New features: `Pr_Error`, `Doppler_L1`, `Cnr_L2`, `Phase_L1_flag`
- Accurate ray-tracing labels (not a simple SNR threshold)
- 74k real samples (not limited to 1k)
- Window gap detection — resets on large time gaps
- Per-SV feature normalisation


In [ ]:
# Install dependencies
!pip install scikit-learn -q
print('Dependencies ready.')

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score
)
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
class Config:
    # File paths — update after uploading to Colab
    TRAIN_PATH = '/content/GNSS_raw_train.csv'
    TEST_PATH  = '/content/GNSS_raw_test.csv'

    # Window
    WINDOW_SIZE  = 10    # epochs per window
    STRIDE       = 1
    MAX_TIME_GAP = 10    # seconds — window resets if gap exceeds this

    # Features — 11 total (v2 had 7)
    N_FEATURES = 11
    # snr_norm, delta_snr, pr_error_norm, delta_pr,
    # doppler_norm, delta_doppler, phase_flag,
    # cnr_l2_norm, sv_type, sin_sv, cos_sv

    # Model
    D_MODEL     = 64
    N_HEADS     = 4
    N_LAYERS    = 4
    D_FF        = 256
    DROPOUT     = 0.1
    NUM_CLASSES = 2

    # Training
    EPOCHS       = 40
    BATCH_SIZE   = 128
    LR           = 1e-3
    WEIGHT_DECAY = 1e-4
    FOCAL_GAMMA  = 2.0
    LABEL_SMOOTH = 0.05
    PATIENCE     = 10

    TEST_SIZE         = 0.15
    VAL_SIZE          = 0.15
    DEFAULT_THRESHOLD = 0.5

cfg = Config()
print('Config loaded.')
print(f'  Features : {cfg.N_FEATURES}')
print(f'  Window   : {cfg.WINDOW_SIZE} epochs')

In [ ]:
# ── Step 1: Data Loading & Feature Engineering ────────────────────────────────

# Constellation code to integer mapping
CONST_MAP = {'G': 0, 'R': 1, 'E': 2, 'C': 3}  # GPS, GLONASS, Galileo, BeiDou

def load_and_engineer(path: str, is_train: bool = True) -> pd.DataFrame:
    """
    Load CSV and build complete feature matrix.
    New features compared to v2:
      - pr_error_norm  : pseudorange error, normalised
      - delta_pr       : Pr_Error change between epochs
      - doppler_norm   : Doppler, normalised
      - delta_doppler  : Doppler change between epochs
      - phase_flag     : whether Phase_L1 is available (0/1)
      - cnr_l2_norm    : CNR L2, normalised
      - sv_type        : constellation type (0-3)
    """
    print(f'Loading {path} ...')
    df = pd.read_csv(path)
    df = df.dropna(subset=['GPS_Time(s)', 'Satelite_Code', 'Cnr_L1']).copy()

    # Constellation type
    df['sv_type'] = df['Satelite_Code'].str[0].map(CONST_MAP).fillna(0).astype(float)

    # SV number — circular encoding
    df['sv_num'] = df['Satelite_Code'].str[1:].apply(
        lambda x: int(x) if x.isdigit() else 0
    ).astype(float)
    df['sin_sv'] = np.sin(2 * np.pi * df['sv_num'] / 40.0)
    df['cos_sv'] = np.cos(2 * np.pi * df['sv_num'] / 40.0)

    # Phase availability flag
    df['phase_flag'] = (df['Phase_L1'] != 0).astype(float)

    # Sort
    df = df.sort_values(['Satelite_Code', 'GPS_Time(s)']).reset_index(drop=True)

    frames = []
    for sv, grp in df.groupby('Satelite_Code'):
        grp = grp.copy().reset_index(drop=True)

        # Time gap detection — reset delta features at large gaps
        time_diff = grp['GPS_Time(s)'].diff().fillna(0)
        gap_mask  = (time_diff > cfg.MAX_TIME_GAP).astype(float)

        # CNR L1: per-SV normalisation
        mean_cnr = grp['Cnr_L1'].mean()
        std_cnr  = grp['Cnr_L1'].std() + 1e-8
        grp['snr_norm'] = (grp['Cnr_L1'] - mean_cnr) / std_cnr

        # Delta SNR (reset at gaps)
        delta_snr = grp['Cnr_L1'].diff().fillna(0)
        delta_snr[gap_mask == 1] = 0.0
        grp['delta_snr'] = delta_snr

        # Pr_Error normalisation — clip outliers first
        pr = grp['Pr_Error'].clip(-100, 100)
        grp['pr_error_norm'] = (pr - pr.mean()) / (pr.std() + 1e-8)

        # Delta Pr_Error
        delta_pr = pr.diff().fillna(0)
        delta_pr[gap_mask == 1] = 0.0
        grp['delta_pr'] = delta_pr / (pr.std() + 1e-8)

        # Doppler normalisation
        dop = grp['Doppler_L1']
        grp['doppler_norm'] = (dop - dop.mean()) / (dop.std() + 1e-8)

        # Delta Doppler
        delta_dop = dop.diff().fillna(0)
        delta_dop[gap_mask == 1] = 0.0
        grp['delta_doppler'] = delta_dop / (dop.std() + 1e-8)

        # CNR L2 normalisation
        cnr2 = grp['Cnr_L2'].fillna(0)
        grp['cnr_l2_norm'] = (cnr2 - cnr2.mean()) / (cnr2.std() + 1e-8)

        # Time gap flag as feature
        grp['gap_flag'] = gap_mask

        frames.append(grp)

    df_feat = pd.concat(frames).sort_values(
        ['Satelite_Code', 'GPS_Time(s)']
    ).reset_index(drop=True)

    if is_train:
        los  = (df_feat['Label'] == 0).sum()
        nlos = (df_feat['Label'] == 1).sum()
        print(f'Samples: {len(df_feat):,}  |  '
              f'LOS: {los:,} ({los/len(df_feat)*100:.1f}%)  '
              f'NLOS: {nlos:,} ({nlos/len(df_feat)*100:.1f}%)')
    else:
        print(f'Test samples: {len(df_feat):,}')

    return df_feat


# Feature columns
FEATURE_COLS = [
    'snr_norm',       # CNR L1 normalised
    'delta_snr',      # CNR L1 delta
    'pr_error_norm',  # pseudorange error normalised  — new
    'delta_pr',       # Pr_Error delta                — new
    'doppler_norm',   # Doppler normalised             — new
    'delta_doppler',  # Doppler delta                 — new
    'phase_flag',     # Phase L1 availability flag    — new
    'cnr_l2_norm',    # CNR L2 normalised             — new
    'sv_type',        # constellation type
    'sin_sv',         # circular SV encoding
    'cos_sv',
]
assert len(FEATURE_COLS) == cfg.N_FEATURES, \
    f'N_FEATURES mismatch: {len(FEATURE_COLS)} vs {cfg.N_FEATURES}'

print('Data utilities defined.')

In [ ]:
# ── Step 2: Window Dataset ────────────────────────────────────────────────────

class GNSSWindowDataset(Dataset):
    """
    Sliding-window dataset with time-gap detection.
    Windows that span a large time gap are skipped.
    """
    def __init__(self, df: pd.DataFrame,
                 feature_cols: list,
                 window_size: int = 10,
                 stride: int = 1,
                 is_train: bool = True):
        self.windows = []
        self.labels  = []
        self.indices = []  # for Kaggle submission

        skipped_gap = 0

        for sv, grp in df.groupby('Satelite_Code'):
            grp   = grp.sort_values('GPS_Time(s)').reset_index(drop=True)
            feats = grp[feature_cols].values.astype(np.float32)
            gaps  = grp['gap_flag'].values

            if is_train:
                labels = grp['Label'].values

            n = len(grp)
            for start in range(0, n - window_size + 1, stride):
                end = start + window_size

                # Skip window if it contains a large time gap
                if gaps[start + 1:end].any():
                    skipped_gap += 1
                    continue

                win = feats[start:end]  # (W, F)
                self.windows.append(win)

                if is_train:
                    # Label = 1 if more than 30% of window epochs are NLOS
                    lab = int(labels[start:end].mean() >= 0.30)
                    self.labels.append(lab)
                else:
                    # Store index of last epoch in window for submission
                    self.indices.append(grp.index[end - 1])

        self.windows = np.array(self.windows, dtype=np.float32)

        if is_train:
            self.labels = np.array(self.labels, dtype=np.int64)
            print(f'Windows: {len(self.windows):,}  '
                  f'(skipped {skipped_gap:,} gap windows)  '
                  f'LOS={(self.labels==0).sum():,}  '
                  f'NLOS={(self.labels==1).sum():,}')
        else:
            print(f'Test windows: {len(self.windows):,}  '
                  f'(skipped {skipped_gap:,} gap windows)')

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.windows[idx])
        if hasattr(self, 'labels'):
            y = torch.tensor(self.labels[idx], dtype=torch.long)
            return x, y
        return x


def make_weighted_sampler(labels: np.ndarray) -> WeightedRandomSampler:
    counts   = np.bincount(labels)
    weights  = 1.0 / counts
    sample_w = weights[labels]
    return WeightedRandomSampler(
        torch.from_numpy(sample_w).float(),
        num_samples=len(sample_w),
        replacement=True
    )

print('Dataset class defined.')

In [ ]:
# ── Step 3: TopConNet Architecture ────────────────────────────────────────────

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * -(np.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])


class TopConNet(nn.Module):
    """
    TopConNet v3 — Transformer Encoder for GNSS LOS/NLOS classification.
    Input: 11 features per epoch (v2 had 7).
    """
    def __init__(self,
                 n_features:  int   = 11,
                 d_model:     int   = 64,
                 n_heads:     int   = 4,
                 n_layers:    int   = 4,
                 d_ff:        int   = 256,
                 dropout:     float = 0.1,
                 num_classes: int   = 2):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(n_features, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
        )
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        self.pos_enc = PositionalEncoding(d_model, dropout=dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=n_layers
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, num_classes),
        )

    def forward(self, x):
        B   = x.size(0)
        x   = self.input_proj(x)
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)
        x   = self.pos_enc(x)
        x   = self.transformer(x)
        return self.classifier(x[:, 0, :])

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Quick architecture check
_m = TopConNet(n_features=cfg.N_FEATURES)
_x = torch.randn(4, cfg.WINDOW_SIZE, cfg.N_FEATURES)
print(f'Input : {_x.shape}  ->  Output: {_m(_x).shape}')
print(f'Params: {_m.count_params():,}')
del _m, _x

In [ ]:
# ── Step 4: Focal Loss + Training Loop ────────────────────────────────────────

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, label_smoothing=0.05, weight=None):
        super().__init__()
        self.gamma           = gamma
        self.label_smoothing = label_smoothing
        self.weight          = weight

    def forward(self, logits, targets):
        nc = logits.size(1)
        with torch.no_grad():
            smooth = torch.zeros_like(logits)
            smooth.fill_(self.label_smoothing / (nc - 1))
            smooth.scatter_(1, targets.unsqueeze(1), 1.0 - self.label_smoothing)
        log_p = torch.log_softmax(logits, dim=-1)
        p_t   = (log_p.exp() * smooth).sum(dim=-1)
        focal = (1 - p_t) ** self.gamma
        loss  = -(focal * (log_p * smooth).sum(dim=-1))
        if self.weight is not None:
            loss = loss * self.weight[targets]
        return loss.mean()


def train_epoch(model, loader, criterion, optimizer, scheduler, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item() * y.size(0)
        correct    += (logits.argmax(1) == y).sum().item()
        total      += y.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    for x, y in loader:
        x, y   = x.to(device), y.to(device)
        logits = model(x)
        loss   = criterion(logits, y)
        total_loss += loss.item() * y.size(0)
        probs  = torch.softmax(logits, dim=1)[:, 1]
        correct += (logits.argmax(1) == y).sum().item()
        total   += y.size(0)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(y.cpu().numpy())
    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    auc = roc_auc_score(all_labels, all_probs) \
          if len(np.unique(all_labels)) > 1 else 0.0
    return total_loss / total, correct / total, auc, all_probs, all_labels


def train(model, train_loader, val_loader, cfg, device, class_weights=None):
    wt = torch.tensor(class_weights, dtype=torch.float).to(device) \
         if class_weights is not None else None
    criterion = FocalLoss(
        gamma=cfg.FOCAL_GAMMA,
        label_smoothing=cfg.LABEL_SMOOTH,
        weight=wt
    )
    optimizer = optim.AdamW(
        model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY
    )
    total_steps = len(train_loader) * cfg.EPOCHS
    scheduler   = CosineAnnealingWarmRestarts(optimizer, T_0=total_steps // 4)

    history = {'train_loss': [], 'val_loss': [],
               'train_acc':  [], 'val_acc':  [], 'val_auc': []}
    best_auc, best_state = 0.0, None
    patience_cnt = 0

    for epoch in range(1, cfg.EPOCHS + 1):
        tr_loss, tr_acc = train_epoch(
            model, train_loader, criterion, optimizer, scheduler, device
        )
        va_loss, va_acc, va_auc, _, _ = eval_epoch(
            model, val_loader, criterion, device
        )
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(va_acc)
        history['val_auc'].append(va_auc)

        print(f'Epoch {epoch:3d}/{cfg.EPOCHS} '
              f'tr_loss={tr_loss:.4f} tr_acc={tr_acc*100:.2f}% '
              f'va_loss={va_loss:.4f} va_acc={va_acc*100:.2f}% '
              f'va_auc={va_auc:.4f}')

        if va_auc > best_auc:
            best_auc   = va_auc
            best_state = {k: v.cpu().clone()
                          for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= cfg.PATIENCE:
                print(f'Early stop at epoch {epoch} (best AUC={best_auc:.4f})')
                break

    model.load_state_dict(best_state)
    return history

print('Training utilities defined.')

In [ ]:
# ── Step 5: Temperature Scaling + Threshold Tuning ────────────────────────────

class TemperatureScaling(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model       = model
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, x):
        return self.model(x) / self.temperature

    @torch.no_grad()
    def fit(self, val_loader, device):
        self.to(device)
        optimizer = optim.LBFGS([self.temperature], lr=0.01, max_iter=50)
        logits_list, labels_list = [], []
        self.model.eval()
        for x, y in val_loader:
            with torch.no_grad():
                logits_list.append(self.model(x.to(device)).cpu())
                labels_list.append(y)
        logits_all = torch.cat(logits_list)
        labels_all = torch.cat(labels_list)

        def closure():
            optimizer.zero_grad()
            loss = nn.CrossEntropyLoss()(
                logits_all.to(device) / self.temperature,
                labels_all.to(device)
            )
            loss.backward()
            return loss
        optimizer.step(closure)
        print(f'Temperature calibrated: T={self.temperature.item():.4f}')


def tune_threshold_roc(probs, labels):
    fpr, tpr, thresholds = roc_curve(labels, probs)
    f1s = [f1_score(labels, (probs >= t).astype(int), zero_division=0)
           for t in thresholds]
    best_t = thresholds[np.argmax(f1s)]
    print(f'Best threshold: {best_t:.4f}  (F1={max(f1s):.4f})')
    return float(best_t)


@torch.no_grad()
def evaluate_model(model, loader, threshold, device):
    model.eval()
    all_probs, all_labels = [], []
    for x, y in loader:
        probs = torch.softmax(model(x.to(device)), 1)[:, 1]
        all_probs.append(probs.cpu().numpy())
        all_labels.append(y.numpy())
    probs  = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    preds  = (probs >= threshold).astype(int)
    print(f'\nTest  Accuracy : {(preds==labels).mean()*100:.2f}%')
    print(f'      ROC-AUC  : {roc_auc_score(labels, probs):.4f}')
    print(f'\n{classification_report(labels, preds, target_names=["LOS","NLOS"])}')
    return probs, labels

print('Calibration utilities defined.')

In [ ]:
# ── Step 6: Visualisation ─────────────────────────────────────────────────────

def plot_training(history):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(history['train_loss'], label='train')
    axes[0].plot(history['val_loss'],   label='val')
    axes[0].set_title('Loss');  axes[0].legend()
    axes[1].plot([a * 100 for a in history['train_acc']], label='train')
    axes[1].plot([a * 100 for a in history['val_acc']],   label='val')
    axes[1].set_title('Accuracy (%)');  axes[1].legend()
    axes[2].plot(history['val_auc'])
    axes[2].set_title('Val ROC-AUC')
    plt.tight_layout();  plt.show()


def plot_confusion(labels, preds):
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap='Blues');  plt.colorbar(im)
    ax.set_xticks([0, 1]);  ax.set_yticks([0, 1])
    ax.set_xticklabels(['LOS', 'NLOS'])
    ax.set_yticklabels(['LOS', 'NLOS'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    fontsize=14,
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')
    ax.set_ylabel('True');  ax.set_xlabel('Predicted')
    ax.set_title('Confusion Matrix')
    plt.tight_layout();  plt.show()


def plot_roc(labels, probs):
    fpr, tpr, _ = roc_curve(labels, probs)
    auc = roc_auc_score(labels, probs)
    plt.figure(figsize=(5, 4))
    plt.plot(fpr, tpr, label=f'AUC={auc:.4f}')
    plt.plot([0, 1], [0, 1], '--', color='gray')
    plt.xlabel('FPR');  plt.ylabel('TPR')
    plt.title('ROC Curve');  plt.legend()
    plt.tight_layout();  plt.show()

print('Visualisation utilities defined.')

In [ ]:
# ── Step 7: Main Pipeline ─────────────────────────────────────────────────────

def main():
    from torch.utils.data import Subset

    # 1. Load & engineer
    print('[1/6] Loading data...')
    df_train = load_and_engineer(cfg.TRAIN_PATH, is_train=True)

    # 2. Build dataset
    print('\n[2/6] Building windows...')
    ds = GNSSWindowDataset(
        df_train, FEATURE_COLS,
        window_size=cfg.WINDOW_SIZE,
        stride=cfg.STRIDE,
        is_train=True
    )

    # 3. Train / val / test split
    idx = np.arange(len(ds))
    lbl = ds.labels
    idx_tr, idx_tmp, l_tr, _ = train_test_split(
        idx, lbl,
        test_size=cfg.TEST_SIZE + cfg.VAL_SIZE,
        stratify=lbl, random_state=SEED
    )
    idx_va, idx_te = train_test_split(
        idx_tmp, test_size=0.5,
        stratify=_, random_state=SEED
    )
    print(f'\nSplit -> train={len(idx_tr):,}  val={len(idx_va):,}  test={len(idx_te):,}')

    train_loader = DataLoader(
        Subset(ds, idx_tr), batch_size=cfg.BATCH_SIZE,
        sampler=make_weighted_sampler(l_tr)
    )
    val_loader  = DataLoader(Subset(ds, idx_va), batch_size=cfg.BATCH_SIZE)
    test_loader = DataLoader(Subset(ds, idx_te), batch_size=cfg.BATCH_SIZE)

    # 4. Build model
    print('\n[3/6] Building model...')
    model = TopConNet(
        n_features=cfg.N_FEATURES,  d_model=cfg.D_MODEL,
        n_heads=cfg.N_HEADS,         n_layers=cfg.N_LAYERS,
        d_ff=cfg.D_FF,               dropout=cfg.DROPOUT,
        num_classes=cfg.NUM_CLASSES
    ).to(DEVICE)
    print(f'  Parameters: {model.count_params():,}')

    # 5. Train
    print('\n[4/6] Training...')
    class_counts  = np.bincount(l_tr)
    class_weights = len(l_tr) / (cfg.NUM_CLASSES * class_counts)
    history = train(model, train_loader, val_loader, cfg, DEVICE, class_weights)

    # 6. Calibrate
    print('\n[5/6] Calibrating...')
    cal_model = TemperatureScaling(model)
    cal_model.fit(val_loader, DEVICE)
    _, _, _, val_probs, val_labels = eval_epoch(
        cal_model, val_loader,
        FocalLoss(cfg.FOCAL_GAMMA, cfg.LABEL_SMOOTH), DEVICE
    )
    best_thr = tune_threshold_roc(val_probs, val_labels)
    cfg.DEFAULT_THRESHOLD = best_thr

    # 7. Evaluate on test set
    print('\n[6/6] Final evaluation...')
    test_probs, test_labels = evaluate_model(
        cal_model, test_loader, best_thr, DEVICE
    )

    # 8. Plots
    plot_training(history)
    plot_confusion(test_labels, (test_probs >= best_thr).astype(int))
    plot_roc(test_labels, test_probs)

    # 9. Save
    save_path = '/content/topconnet_v3.pt'
    torch.save({
        'model_state':  cal_model.state_dict(),
        'config':       vars(cfg),
        'threshold':    best_thr,
        'feature_cols': FEATURE_COLS,
    }, save_path)
    print(f'\nModel saved -> {save_path}')

    return cal_model, best_thr


model, threshold = main()

In [ ]:
# ── Step 8: Kaggle Submission ─────────────────────────────────────────────────

@torch.no_grad()
def generate_submission(model, threshold, cfg):
    print('Building test predictions...')
    df_test = load_and_engineer(cfg.TEST_PATH, is_train=False)

    test_ds = GNSSWindowDataset(
        df_test, FEATURE_COLS,
        window_size=cfg.WINDOW_SIZE,
        stride=cfg.STRIDE,
        is_train=False
    )
    test_loader = DataLoader(test_ds, batch_size=cfg.BATCH_SIZE)

    model.eval()
    all_probs = []
    for x in test_loader:
        probs = torch.softmax(model(x.to(DEVICE)), 1)[:, 1]
        all_probs.append(probs.cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_preds = (all_probs >= threshold).astype(int)

    # Build submission from sample file
    sample_sub = pd.read_csv('/content/sample_submission.csv')
    sub = sample_sub.copy()
    sub.loc[:len(all_preds) - 1, 'Predict'] = all_preds[:len(sub)]

    out_path = '/content/submission_topconnet_v3.csv'
    sub.to_csv(out_path, index=False)
    print(f'Saved -> {out_path}')
    print(f'LOS: {(all_preds == 0).sum():,}  NLOS: {(all_preds == 1).sum():,}')
    return sub


sub = generate_submission(model, threshold, cfg)